# Populate World-Camera Calibration Data

The goal of this notebook is to **generate the calibration assets under `data/` from the most primary available sources on Dropbox**. For camera measurements, that means raw sensor chunks and their metadata. For external measurements, that means the original instrument export or reference table. Checked-in TIFF and MAT outputs are never source material and must not be copied into place. They may be used only to validate a newly generated result.

This notebook is currently an **implementation scaffold**, not yet a complete one-click rebuild. Every write operation is disabled by default. Set a `RUN_*` flag to `True` only after confirming the corresponding Dropbox source and selection rule. Existing outputs are preserved unless that section's `OVERWRITE_*` flag is also enabled. Any use of a current output to recover a historical frame index is a temporary migration aid; the finished workflow must store the raw selection rule or stable raw index and work when `data/` starts empty.

Deprecated calibration artifacts, hidden operating-system files, generated implementation caches, and previously derived copies on Dropbox are intentionally outside the source boundary.

## Build outline and handoff

The intended execution order is:

1. **Declare primary sources.** Configure one Dropbox root and record the relative source path, source type, and selection rule for every output. Do not point at an existing generated TIFF or MAT file.
2. **Run a read-only preflight.** Confirm that each source exists, contains the expected raw chunk/metadata files, and has the expected recording or measurement structure before writing anything.
3. **Materialize `data/` inputs.** Extract raw frames directly when stable global indices are known. Reconstruct a temporary video only when the authoritative selection is expressed in video time or requires the normal frame-gap handling. Parse original instrument exports and reference tables into MAT files with explicit schemas.
4. **Run the calibration definitions.** Feed the newly generated `data/` inputs into the MATLAB and Python definition routines that write the corresponding files under `derived/`. Interactive decisions, such as checkerboard acceptance in Camera Calibrator, must be documented and preserved as reproducible selections where possible.
5. **Validate a clean rebuild.** Check file counts, array shapes, variable names, finite ranges, and important numerical summaries. Comparison with the checked-in assets is allowed here, but never during generation.

### Asset-by-asset plan

| Destination generated under `data/` | Most primary source | Generation |
| --- | --- | --- |
| `exampleWorldCameraImages/*.tiff` and paired `*_AGCandMS_*.mat` files | FLIC_2001 indoor/outdoor and planetarium raw world/minispect chunks | Extract the selected raw Bayer frames directly by global index in the chunks (non-gap-filled), then generate one paired MAT file per TIFF containing that frame's AGC settings and the nearest minispect sample on the shared clock. |
| `fisheyeLensCalibration/intrinsics_calibration_images/` | Raw checkerboard-calibration chunks | Extract the accepted frames. The reader must then use MATLAB's built-in Camera Calibrator app to fit the model, save the session, and export `derived/arducamB0392cameraIntrinsics.mat`; that interactive step is not automated here. |
| `flatFieldingFunction/rawFrames/` | Raw Fels Planetarium chunks | Reconstruct the recording with the documented gap and gain handling, extract the 36 documented time samples, then run `defineFlatFieldingFunction.m` to create `derived/flatFieldingFunction.mat` |
| `radiometricCorrectionRGB/rawFrames/` and cloudy-sky SPD | Raw simultaneous camera chunks plus the original PR670 export | Extract the ten uncorrected Bayer frames, parse the PR670 measurement, then run `defineRadiometricWeights.m` to create `derived/radiometricCorrectionRGB.mat` |
| `darkSignal/<AGC state>/*.tiff` | Covered-camera raw chunks for five fixed AGC states | Select ten documented raw frames per state, then run `defineDarkSignal.m` to create `derived/darkSignal.mat` |
| `camera_linearity_ND0_ND0p4_rgb_means.mat` | Original radiometric-calibration collection outputs | Run the existing conversion and camera-linearity analysis, then use the result in `defineFullWellCapacityEffect.m` |
| `empircalAGCAndIlluminance.mat` | Raw GKA world/minispect recordings | Derive `derived/MSIlluminanceToAGCLag.mat`, build the aligned point cloud, and compare it with the integrating-sphere lookup saved by `defineAGCToMeanLuminance.m` in `derived/cameraScoreToAverageLuminance.mat` |
| `agc_empirical_kernels.mat` | AGC simulation parameters and algorithm | Generate `derived/MSIlluminanceToAGCKernel.mat` with `defineMSIlluminanceToAGCKernel.py`; `defineMSIlluminanceToAGCLag.py` loads that standalone artifact |
| `ASM7341_spectralSensitivity.mat` and `IMX219_spectralSensitivity.mat` | Original manufacturer spreadsheet / published reference table | Parse the primary tables into stable MAT schemas used by downstream calibration code |


# Utility functions

Run this cell first. It defines the shared imports, project paths, input preflight checks, safe data-output helpers, raw world-frame access, README block updates, and lazy video I/O used by all later stages.

In [ ]:
from __future__ import annotations

import importlib
import re
import shutil
import sys
import tempfile
from collections.abc import Mapping
from pathlib import Path
from typing import Any

import cv2
import numpy as np
from scipy.io import savemat


# Assume Jupyter was launched from code/defineWorldCameraCalibration/dataPrep.
NOTEBOOK_DIR: Path = Path.cwd().resolve()
PROJECT_ROOT: Path = NOTEBOOK_DIR.parents[2]
DATA_ROOT: Path = PROJECT_ROOT / "data"
CHUNK_IO_PATH: Path = (
    PROJECT_ROOT / "code" / "library" / "matlabIO" / "python_libraries"
)
SENSOR_UTILITY_PATH: Path = PROJECT_ROOT / "code" / "library" / "sensor_utility"
for library_path in (CHUNK_IO_PATH, SENSOR_UTILITY_PATH):
    if not library_path.is_dir():
        raise FileNotFoundError(f"Required Python library directory does not exist: {library_path}")
    if str(library_path) not in sys.path:
        sys.path.insert(0, str(library_path))
import chunk_io
chunk_io: Any = importlib.reload(chunk_io)


# -----------------------------------------------------------------------------
# Lazy video dependency
# Keep video_io and its MATLAB dependency out of memory until a video-based
# extraction stage is explicitly enabled.
# -----------------------------------------------------------------------------
def load_video_io() -> Any:
    """Import the heavier video/MATLAB utility only when a stage needs it."""
    import video_io

    return importlib.reload(video_io)


# -----------------------------------------------------------------------------
# Input preflight
# Check every configured directory or file together before a stage starts any
# expensive processing, and report all missing inputs in one error.
# -----------------------------------------------------------------------------
def require_existing_directories(paths_by_name: Mapping[str, Path]) -> None:
    """Raise one error listing every configured input that is not a directory."""
    missing: list[tuple[str, Path]] = [
        (name, path) for name, path in paths_by_name.items() if not path.is_dir()
    ]
    if missing:
        details: str = "\n".join(f"- {name}: {path}" for name, path in missing)
        raise FileNotFoundError(f"Required input directories do not exist:\n{details}")


def require_existing_files(paths_by_name: Mapping[str, Path]) -> None:
    """Raise one error listing every configured input that is not a file."""
    missing: list[tuple[str, Path]] = [
        (name, path) for name, path in paths_by_name.items() if not path.is_file()
    ]
    if missing:
        details: str = "\n".join(f"- {name}: {path}" for name, path in missing)
        raise FileNotFoundError(f"Required input files do not exist:\n{details}")


# -----------------------------------------------------------------------------
# Safe output and TIFF writing
# Restrict writes to children of data/, require explicit replacement of existing
# directories, and centralize sequential uncompressed TIFF naming.
# -----------------------------------------------------------------------------
def assert_safe_data_target(target: Path) -> Path:
    """Require an output target to be below data/, never data/ itself."""
    resolved_target: Path = target.resolve()
    resolved_data_root: Path = DATA_ROOT.resolve()
    if (
        resolved_target == resolved_data_root
        or resolved_data_root not in resolved_target.parents
    ):
        raise ValueError(f"Refusing an unsafe output target: {resolved_target}")
    return resolved_target


def prepare_output_directory(target: Path, *, overwrite: bool) -> Path:
    """Prepare an empty data subdirectory without replacing files implicitly."""
    target = assert_safe_data_target(target)
    target.mkdir(parents=True, exist_ok=True)
    existing_items: list[Path] = list(target.iterdir())
    if existing_items and not overwrite:
        raise FileExistsError(
            f"{target} is not empty. Set this stage's overwrite flag to replace it."
        )
    if overwrite:
        for item in existing_items:
            if item.is_dir():
                shutil.rmtree(item)
            else:
                item.unlink()
    return target


def write_tiff_stack(
    frames: np.ndarray | list[np.ndarray], output_dir: Path, *, overwrite: bool
) -> None:
    """Write frames as sequential zero-based uncompressed TIFF files."""
    output_dir = prepare_output_directory(output_dir, overwrite=overwrite)
    for frame_index, frame in enumerate(frames):
        output_path: Path = output_dir / f"{frame_index}.tiff"
        written: bool = cv2.imwrite(
            str(output_path), frame, [cv2.IMWRITE_TIFF_COMPRESSION, 1]
        )
        if not written:
            raise IOError(f"OpenCV could not write {output_path}")
    print(f"Wrote {len(frames)} frames to {output_dir}")


# -----------------------------------------------------------------------------
# Direct raw world-frame access
# Reuse chunk_io's natural chunk ordering to count frames, extract stable global
# indices without a video codec, and discover indices from existing raw TIFFs.
# -----------------------------------------------------------------------------
def raw_world_frame_count(raw_chunks: Path) -> int:
    """Count frames physically present in naturally ordered world chunks."""
    require_existing_directories({"world-camera raw chunks": raw_chunks})
    chunk_pairs: list[tuple[str, str]] = chunk_io.group_sensors_files(
        str(raw_chunks)
    )["W"]
    if not chunk_pairs:
        raise FileNotFoundError(f"No world frame chunks found in {raw_chunks}")
    return sum(
        int(np.load(frame_path, mmap_mode="r").shape[0])
        for _, frame_path in chunk_pairs
    )


def extract_raw_world_frames(
    raw_chunks: Path, frame_indices: list[int]
) -> np.ndarray:
    """Extract raw frames by zero-based global index without making a video."""
    require_existing_directories({"world-camera raw chunks": raw_chunks})
    if not frame_indices or any(index < 0 for index in frame_indices):
        raise ValueError("frame_indices must contain nonnegative global indices.")
    chunk_pairs: list[tuple[str, str]] = chunk_io.group_sensors_files(
        str(raw_chunks)
    )["W"]
    if not chunk_pairs:
        raise FileNotFoundError(f"No world frame chunks found in {raw_chunks}")
    requested_positions: dict[int, list[int]] = {}
    for output_position, frame_index in enumerate(frame_indices):
        requested_positions.setdefault(int(frame_index), []).append(output_position)
    extracted: list[np.ndarray | None] = [None] * len(frame_indices)
    global_offset: int = 0
    for _, frame_path in chunk_pairs:
        chunk_frames: np.memmap = np.load(frame_path, mmap_mode="r")
        chunk_stop: int = global_offset + len(chunk_frames)
        for frame_index, output_positions in requested_positions.items():
            if global_offset <= frame_index < chunk_stop:
                frame: np.ndarray = np.asarray(
                    chunk_frames[frame_index - global_offset]
                ).copy()
                for output_position in output_positions:
                    extracted[output_position] = frame
        global_offset = chunk_stop
        if all(frame is not None for frame in extracted):
            break
    missing_indices: list[int] = [
        frame_indices[index]
        for index, frame in enumerate(extracted)
        if frame is None
    ]
    if missing_indices:
        raise IndexError(f"Raw frame indices exceed available chunks: {missing_indices}")
    return np.stack([frame for frame in extracted if frame is not None])


def discover_reference_frame_indices(
    reference_paths: Mapping[str, Path], raw_chunks_by_name: Mapping[str, Path]
) -> dict[str, int]:
    """Match existing raw TIFFs to source chunks and return global indices."""
    require_existing_files(reference_paths)
    require_existing_directories(raw_chunks_by_name)
    discovered: dict[str, int] = {}
    for name, reference_path in reference_paths.items():
        target: np.ndarray | None = cv2.imread(
            str(reference_path), cv2.IMREAD_UNCHANGED
        )
        if target is None:
            raise FileNotFoundError(reference_path)
        frame_index: int | None = chunk_io.find_frame_index(
            str(raw_chunks_by_name[name]), target, verbose=True
        )
        if frame_index is None:
            raise ValueError(f"Could not find {name} in {raw_chunks_by_name[name]}")
        discovered[name] = int(frame_index)
    return discovered


# -----------------------------------------------------------------------------
# Generated README sections
# Replace only a named marker-delimited block so generated provenance can be
# refreshed without disturbing the README's hand-written scientific notes.
# -----------------------------------------------------------------------------
def update_generated_readme_section(
    readme_path: Path, section_name: str, markdown: str
) -> None:
    """Replace one marker-delimited generated section of a data README."""
    readme_path = assert_safe_data_target(readme_path)
    require_existing_files({f"{section_name} README": readme_path})
    start_marker: str = f"<!-- populateData:{section_name}:start -->"
    end_marker: str = f"<!-- populateData:{section_name}:end -->"
    text: str = readme_path.read_text(encoding="utf-8")
    if text.count(start_marker) != 1 or text.count(end_marker) != 1:
        raise ValueError(f"README markers for {section_name!r} are missing or duplicated.")
    before, remainder = text.split(start_marker, 1)
    after: str = remainder.split(end_marker, 1)[1]
    replacement: str = (
        f"{start_marker}\n{markdown.rstrip()}\n{end_marker}"
    )
    readme_path.write_text(before + replacement + after, encoding="utf-8")

# `data/exampleWorldCameraImages/`

**What this is:** A small curated gallery used to inspect representative world-camera images from indoor, outdoor, and planetarium conditions. The current archive contains three indoor images, two outdoor images, and one planetarium image. All six images are completely raw, with no processing applied.

**Where it comes from:** Frames were manually selected from the FLIC_2001 `walkIndoor` and `walkOutdoor` recordings and from the planetarium recording. The visual choice is not algorithmic, so the current TIFFs serve once as reference targets for `chunk_io.find_frame_index`. The discovered zero-based raw indices are written into the directory README and become the reproducible selection.

**Output:** `data/exampleWorldCameraImages/*.tiff`.

## 1. Discover and record existing frame indices

Run this cell while the curated TIFFs and raw recordings both exist. It finds each TIFF's exact raw-frame index and records the resulting index table in `data/exampleWorldCameraImages/README.md`.

In [4]:
RUN_EXAMPLE_INDEX_DISCOVERY: bool = True

EXAMPLE_IMAGE_OUTPUT: Path = DATA_ROOT / "exampleWorldCameraImages"
EXPECTED_EXAMPLE_NAMES: set[str] = {
    "indoor_1.tiff",
    "indoor_2.tiff",
    "indoor_3.tiff",
    "outdoor_1.tiff",
    "outdoor_2.tiff",
    "planetarium_1.tiff",
}
EXAMPLE_REFERENCE_PATHS: dict[str, Path] = {
    name: EXAMPLE_IMAGE_OUTPUT / name for name in EXPECTED_EXAMPLE_NAMES
}
EXAMPLE_RAW_CHUNKS_BY_NAME: dict[str, Path] = {
    "indoor_1.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"),
    "indoor_2.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"),
    "indoor_3.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkIndoor/GKA"),
    "outdoor_1.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA"),
    "outdoor_2.tiff": Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026/FLIC_2001/walkOutdoor/GKA"),
    "planetarium_1.tiff":  Path("/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/flatFieldingFunction/planetarium_fielding_function_raw"),
}
EXAMPLE_SOURCE_LABELS: dict[str, str] = {
    "indoor_1.tiff": "FLIC_2001 walkIndoor",
    "indoor_2.tiff": "FLIC_2001 walkIndoor",
    "indoor_3.tiff": "FLIC_2001 walkIndoor",
    "outdoor_1.tiff": "FLIC_2001 walkOutdoor",
    "outdoor_2.tiff": "FLIC_2001 walkOutdoor",
    "planetarium_1.tiff": "Fels Planetarium",
}


# -----------------------------------------------------------------------------
# README writer
# Turn the discovered filename/index mapping into a Markdown table and replace
# only the marker-delimited block, preserving all hand-written README content.
# -----------------------------------------------------------------------------
def update_example_frame_index_readme(frame_indices: dict[str, int]) -> None:
    """Replace only the generated raw-index table in the example-image README."""
    rows: list[str] = [
        "| Frame | Source recording | Zero-based global raw-frame index |",
        "| --- | --- | ---: |",
    ]
    for name in sorted(frame_indices):
        rows.append(f"| `{name}` | {EXAMPLE_SOURCE_LABELS[name]} | {frame_indices[name]} |")
    readme_path: Path = EXAMPLE_IMAGE_OUTPUT / "README.md"
    start_marker: str = "<!-- populateData:example-frame-indices:start -->"
    end_marker: str = "<!-- populateData:example-frame-indices:end -->"
    text: str = readme_path.read_text(encoding="utf-8")
    if text.count(start_marker) != 1 or text.count(end_marker) != 1:
        raise ValueError("The README example-frame-index markers are missing or duplicated.")
    before: str
    remainder: str
    before, remainder = text.split(start_marker, 1)
    after: str = remainder.split(end_marker, 1)[1]
    table: str = "\n".join(rows)
    replacement: str = f"{start_marker}\n{table}\n{end_marker}"
    readme_path.write_text(before + replacement + after, encoding="utf-8")


# -----------------------------------------------------------------------------
# Frame-index discovery
# Load each curated TIFF without changing its pixels, search its corresponding
# raw chunk sequence for an exact match, and return the stable global indices.
# -----------------------------------------------------------------------------
def discover_example_frame_indices() -> dict[str, int]:
    """Match each existing TIFF exactly against its naturally ordered raw chunks."""
    require_existing_files(EXAMPLE_REFERENCE_PATHS)
    require_existing_files({"example-image README": EXAMPLE_IMAGE_OUTPUT / "README.md"})
    require_existing_directories(EXAMPLE_RAW_CHUNKS_BY_NAME)
    discovered: dict[str, int] = {}
    for name, reference_path in sorted(EXAMPLE_REFERENCE_PATHS.items()):
        target: np.ndarray | None = cv2.imread(str(reference_path), cv2.IMREAD_UNCHANGED)
        if target is None:
            raise FileNotFoundError(reference_path)
        frame_index: int | None = chunk_io.find_frame_index(
            str(EXAMPLE_RAW_CHUNKS_BY_NAME[name]), target, verbose=True
        )
        if frame_index is None:
            raise ValueError(f"Could not find {name} in {EXAMPLE_RAW_CHUNKS_BY_NAME[name]}")
        discovered[name] = int(frame_index)
    return discovered


# -----------------------------------------------------------------------------
# Execution gate
# Leave the flag False for ordinary notebook runs. Set it True only when the
# existing TIFFs and all source recordings are available for one-time discovery.
# -----------------------------------------------------------------------------
if RUN_EXAMPLE_INDEX_DISCOVERY:
    discovered_example_indices: dict[str, int] = discover_example_frame_indices()
    update_example_frame_index_readme(discovered_example_indices)
    print("Recorded example indices:", discovered_example_indices)

Searching raw world chunks:   0%|          | 0/12 [00:00<?, ?chunk/s]

Searching raw world chunks:   0%|          | 0/12 [00:00<?, ?chunk/s]

Searching raw world chunks:   0%|          | 0/12 [00:00<?, ?chunk/s]

Searching raw world chunks:   0%|          | 0/12 [00:00<?, ?chunk/s]

Searching raw world chunks:   0%|          | 0/12 [00:00<?, ?chunk/s]

Searching raw world chunks:   0%|          | 0/33 [00:00<?, ?chunk/s]

Recorded example indices: {'indoor_1.tiff': 4777, 'indoor_2.tiff': 42993, 'indoor_3.tiff': 21713, 'outdoor_1.tiff': 4512, 'outdoor_2.tiff': 40615, 'planetarium_1.tiff': 96978}


## 2. Rebuild images and synchronized metadata from the README

Run this cell after the README contains the discovered indices. It regenerates each raw TIFF and a paired MAT file containing the frame, AGC settings, and nearest minispect sample.

In [5]:
RUN_EXAMPLE_FRAME_EXTRACTION: bool = True
OVERWRITE_EXAMPLE_IMAGES: bool = False

# -----------------------------------------------------------------------------
# README schema and MAT-file documentation
# The block regex isolates the generated README section; the row regex extracts
# each TIFF name and integer index. EXAMPLE_MAT_README documents saved variables.
# -----------------------------------------------------------------------------
EXAMPLE_INDEX_BLOCK_PATTERN: re.Pattern[str] = re.compile(
    r"<!-- populateData:example-frame-indices:start -->(?P<table>.*?)"
    r"<!-- populateData:example-frame-indices:end -->",
    re.DOTALL,
)
EXAMPLE_INDEX_ROW_PATTERN: re.Pattern[str] = re.compile(
    r"^\|\s*`(?P<name>[^`]+\.tiff)`\s*\|[^|]*\|\s*(?P<index>\d+)\s*\|$",
    re.MULTILINE,
)
EXAMPLE_MAT_README: str = (
    "Synchronized context for one curated raw world-camera example. "
    "sourceImageFilename names the paired TIFF; globalWorldFrameIndex is its zero-based "
    "index across naturally ordered raw world chunks; worldTimestampSeconds is on the "
    "shared logger clock; worldFrame is the unprocessed Bayer frame; AGCSettings contains "
    "all world-camera metadata settings except timestamp; minispectTimestampSeconds and "
    "minispectValue contain the nearest minispect packet and its parsed AS, TS, LS, and TEMP values."
)


# -----------------------------------------------------------------------------
# README parser
# Read the marker-delimited table with regex and validate it before any raw data
# are opened, so missing, duplicate, or unexpected image rows fail immediately.
# -----------------------------------------------------------------------------
def read_example_frame_indices_from_readme() -> dict[str, int]:
    """Extract the curated TIFF names and global indices from the README table using regex."""
    readme_path: Path = EXAMPLE_IMAGE_OUTPUT / "README.md"
    require_existing_files({"example-image README": readme_path})
    readme_text: str = readme_path.read_text(encoding="utf-8")
    blocks: list[re.Match[str]] = list(EXAMPLE_INDEX_BLOCK_PATTERN.finditer(readme_text))
    if len(blocks) != 1:
        raise ValueError("The README must contain exactly one generated example-frame-index block.")
    matches: list[re.Match[str]] = list(
        EXAMPLE_INDEX_ROW_PATTERN.finditer(blocks[0].group("table"))
    )
    frame_indices: dict[str, int] = {
        match.group("name"): int(match.group("index")) for match in matches
    }
    if len(frame_indices) != len(matches):
        raise ValueError("The README contains duplicate example-frame rows.")
    if set(frame_indices) != EXPECTED_EXAMPLE_NAMES:
        missing: list[str] = sorted(EXPECTED_EXAMPLE_NAMES - set(frame_indices))
        extra: list[str] = sorted(set(frame_indices) - EXPECTED_EXAMPLE_NAMES)
        raise ValueError(f"README example-frame rows do not match the curated set; missing={missing}, extra={extra}")
    return frame_indices


# -----------------------------------------------------------------------------
# Global index to timestamp lookup
# Walk the naturally ordered world chunks and translate a README global index
# into the exact world-camera timestamp needed by chunk_io's sensor lookup.
# -----------------------------------------------------------------------------
def world_timestamp_at_global_frame_index(raw_chunks: Path, global_frame_index: int) -> float:
    """Map a physical global world-frame index to its exact timestamp in seconds."""
    require_existing_directories({"world-camera raw chunks": raw_chunks})
    if global_frame_index < 0:
        raise ValueError("global_frame_index must be nonnegative.")
    global_offset: int = 0
    for metadata_path, value_path in chunk_io.group_sensors_files(str(raw_chunks))["W"]:
        metadata: np.memmap = np.load(metadata_path, mmap_mode="r")
        values: np.memmap = np.load(value_path, mmap_mode="r")
        if len(metadata) != len(values):
            raise ValueError(f"World metadata/value length mismatch: {metadata_path}, {value_path}")
        chunk_stop: int = global_offset + len(values)
        if global_frame_index < chunk_stop:
            local_index: int = global_frame_index - global_offset
            timestamp_nanoseconds: np.generic | float = (
                metadata[local_index] if metadata.ndim == 1 else metadata[local_index, 0]
            )
            return float(timestamp_nanoseconds) / 1e9
        global_offset = chunk_stop
    raise IndexError(f"Global frame index {global_frame_index} exceeds {global_offset} raw frames.")


# -----------------------------------------------------------------------------
# Synchronized sensor retrieval
# Ask chunk_io for the raw world entry at the indexed timestamp and the nearest
# minispect packet; the world entry also carries that frame's complete AGC settings.
# -----------------------------------------------------------------------------
def example_context_at_global_frame_index(
    raw_chunks: Path, global_frame_index: int
) -> dict[str, dict[str, Any]]:
    """Retrieve the indexed raw frame, its AGC settings, and nearest minispect packet."""
    timestamp_seconds: float = world_timestamp_at_global_frame_index(
        raw_chunks, global_frame_index
    )
    context: dict[str, dict[str, Any]] = chunk_io.find_nearest_neighbor(
        str(raw_chunks), timestamp_seconds, sensors=("W", "M")
    )
    if context["W"]["timestamp"] != timestamp_seconds:
        raise ValueError("The indexed world timestamp did not resolve to the same raw frame.")
    return context


# -----------------------------------------------------------------------------
# Paired MAT filename construction
# Convert a curated name such as indoor_3.tiff into the established paired name
# indoor_AGCandMS_03.mat while rejecting filenames outside that convention.
# -----------------------------------------------------------------------------
def example_mat_path(image_name: str) -> Path:
    """Return the established condition_AGCandMS_XX.mat filename for a TIFF."""
    match: re.Match[str] | None = re.fullmatch(
        r"(?P<condition>.+)_(?P<sequence>\d+)\.tiff", image_name
    )
    if match is None:
        raise ValueError(f"Unsupported example-image filename: {image_name}")
    return EXAMPLE_IMAGE_OUTPUT / (
        f"{match.group('condition')}_AGCandMS_{int(match.group('sequence')):02d}.mat"
    )


# -----------------------------------------------------------------------------
# Output generation
# Parse the README, decide which outputs are allowed to be written, retrieve the
# needed contexts, then save missing files without touching protected outputs.
# -----------------------------------------------------------------------------
def populate_example_world_camera_images(*, overwrite: bool = False) -> None:
    """Generate missing TIFF/MAT pairs, replacing existing files only when allowed."""
    frame_indices: dict[str, int] = read_example_frame_indices_from_readme()
    output_paths_by_name: dict[str, tuple[Path, Path]] = {
        name: (EXAMPLE_IMAGE_OUTPUT / name, example_mat_path(name))
        for name in frame_indices
    }
    names_requiring_context: list[str] = [
        name
        for name, output_paths in output_paths_by_name.items()
        if overwrite or any(not path.exists() for path in output_paths)
    ]
    protected_output_count: int = (
        0
        if overwrite
        else sum(
            path.exists()
            for output_paths in output_paths_by_name.values()
            for path in output_paths
        )
    )

    # Raw recordings are required only when at least one output will be written.
    if names_requiring_context:
        require_existing_directories(EXAMPLE_RAW_CHUNKS_BY_NAME)
    contexts: dict[str, dict[str, dict[str, Any]]] = {
        name: example_context_at_global_frame_index(
            EXAMPLE_RAW_CHUNKS_BY_NAME[name], global_frame_index
        )
        for name, global_frame_index in sorted(frame_indices.items())
        if name in names_requiring_context
    }
    EXAMPLE_IMAGE_OUTPUT.mkdir(parents=True, exist_ok=True)
    written_tiff_count: int = 0
    written_mat_count: int = 0
    for name, global_frame_index in sorted(frame_indices.items()):
        if name not in contexts:
            continue
        context: dict[str, dict[str, Any]] = contexts[name]
        image_path, mat_path = output_paths_by_name[name]
        if overwrite or not image_path.exists():
            written: bool = cv2.imwrite(
                str(image_path),
                context["W"]["value"],
                [cv2.IMWRITE_TIFF_COMPRESSION, 1],
            )
            if not written:
                raise IOError(f"OpenCV could not write {image_path}")
            written_tiff_count += 1
        if overwrite or not mat_path.exists():
            savemat(
                mat_path,
                {
                    "README": EXAMPLE_MAT_README,
                    "sourceImageFilename": name,
                    "globalWorldFrameIndex": np.int64(global_frame_index),
                    "worldTimestampSeconds": context["W"]["timestamp"],
                    "worldFrame": context["W"]["value"],
                    "AGCSettings": context["W"]["AGCSettings"],
                    "minispectTimestampSeconds": context["M"]["timestamp"],
                    "minispectValue": context["M"]["value"],
                },
                do_compression=True,
                long_field_names=True,
            )
            written_mat_count += 1
    print(
        f"Generated {written_tiff_count} TIFFs and {written_mat_count} MAT files; "
        f"preserved {protected_output_count} existing outputs in {EXAMPLE_IMAGE_OUTPUT}"
    )


# -----------------------------------------------------------------------------
# Execution gate
# Set RUN_EXAMPLE_FRAME_EXTRACTION to True to generate outputs. Existing files
# remain protected unless OVERWRITE_EXAMPLE_IMAGES is also explicitly enabled.
# -----------------------------------------------------------------------------
if RUN_EXAMPLE_FRAME_EXTRACTION:
    populate_example_world_camera_images(
        overwrite=OVERWRITE_EXAMPLE_IMAGES
    )

FileExistsError: Example outputs already exist. Set OVERWRITE_EXAMPLE_IMAGES to True to replace: indoor_1.tiff, indoor_AGCandMS_01.mat, indoor_2.tiff, indoor_AGCandMS_02.mat, indoor_3.tiff, indoor_AGCandMS_03.mat, outdoor_1.tiff, outdoor_AGCandMS_01.mat, outdoor_2.tiff, outdoor_AGCandMS_02.mat, planetarium_1.tiff, planetarium_AGCandMS_01.mat

# `data/fisheyeLensCalibration/`

**What this is:** The checkerboard images used as input to MATLAB's built-in Camera Calibrator app to estimate focal length, principal point, and fisheye distortion for the ArduCam B0392 IMX219 world camera.

**Where it comes from:** A manually collected checkerboard-calibration raw recording followed by selection and fitting in MATLAB's Camera Calibrator app. The accepted local TIFFs are used once to discover their global raw-frame indices; thereafter the 47-image input set is recreated directly from the raw chunks.

**What it does:** Discovers and records the 47 selected raw indices, then extracts those frames without a video conversion. This notebook stops after producing the input TIFFs.

**Required manual MATLAB step:** Open MATLAB's built-in Camera Calibrator app (for example, by running `cameraCalibrator`), load the TIFFs from `data/fisheyeLensCalibration/intrinsics_calibration_images/`, review checkerboard acceptance, fit the camera model, save the calibration session, and export the resulting intrinsics to `derived/arducamB0392cameraIntrinsics.mat`. These GUI decisions are intentionally left to the reader and are not run by this notebook.

**Notebook output:** `data/fisheyeLensCalibration/intrinsics_calibration_images/`.

In [ ]:
RUN_FISHEYE_INDEX_DISCOVERY: bool = False
RUN_FISHEYE_FRAME_EXTRACTION: bool = False
OVERWRITE_FISHEYE_IMAGES: bool = False
FISHEYE_RAW_CHUNKS: Path | None = "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/FOV_and_intrinsics/intrinsics_calibration/intrinsics_calibration_video"
FISHEYE_OUTPUT: Path = DATA_ROOT / "fisheyeLensCalibration"
FISHEYE_IMAGE_OUTPUT: Path = FISHEYE_OUTPUT / "intrinsics_calibration_images"
FISHEYE_IMAGE_NAMES: list[str] = [f"{index}.tiff" for index in range(47)]
FISHEYE_REFERENCE_PATHS: dict[str, Path] = {
    name: FISHEYE_IMAGE_OUTPUT / name for name in FISHEYE_IMAGE_NAMES
}
# Populate this mapping with the discovery result to make later runs independent
# of the already-extracted reference TIFFs.
FISHEYE_FRAME_INDICES: dict[str, int | None] = {name: None for name in FISHEYE_IMAGE_NAMES}


def update_fisheye_frame_index_readme(frame_indices: dict[str, int]) -> None:
    """Record the accepted checkerboard frames' global raw indices."""
    rows: list[str] = [
        "| Calibration image | Zero-based global raw-frame index |",
        "| --- | ---: |",
    ]
    for name in FISHEYE_IMAGE_NAMES:
        rows.append(f"| `{name}` | {frame_indices[name]} |")
    update_generated_readme_section(
        FISHEYE_OUTPUT / "README.md", "fisheye-frame-indices", "\n".join(rows)
    )


def populate_fisheye_calibration_images(
    raw_chunks: Path, frame_indices: dict[str, int], *, overwrite: bool = False
) -> None:
    """Recreate the accepted checkerboard images directly from raw chunks."""
    require_existing_directories({"fisheye raw chunks": raw_chunks})
    if list(frame_indices) != FISHEYE_IMAGE_NAMES:
        raise ValueError("Fisheye indices must be ordered from 0.tiff through 46.tiff.")
    # Preserve selection order so write_tiff_stack recreates the expected filenames.
    frames: np.ndarray = extract_raw_world_frames(
        raw_chunks, list(frame_indices.values())
    )
    write_tiff_stack(frames, FISHEYE_IMAGE_OUTPUT, overwrite=overwrite)


if RUN_FISHEYE_INDEX_DISCOVERY:
    if FISHEYE_RAW_CHUNKS is None:
        raise ValueError("Set FISHEYE_RAW_CHUNKS before discovering frame indices.")
    require_existing_directories({"fisheye raw chunks": FISHEYE_RAW_CHUNKS})
    require_existing_files(FISHEYE_REFERENCE_PATHS)
    discovered_fisheye_indices: dict[str, int] = discover_reference_frame_indices(
        FISHEYE_REFERENCE_PATHS,
        {name: FISHEYE_RAW_CHUNKS for name in FISHEYE_IMAGE_NAMES},
    )
    FISHEYE_FRAME_INDICES.update(discovered_fisheye_indices)
    update_fisheye_frame_index_readme(discovered_fisheye_indices)
    print("Discovered fisheye indices:", discovered_fisheye_indices)

if RUN_FISHEYE_FRAME_EXTRACTION:
    if FISHEYE_RAW_CHUNKS is None:
        raise ValueError("Set FISHEYE_RAW_CHUNKS before extracting calibration images.")
    require_existing_directories({"fisheye raw chunks": FISHEYE_RAW_CHUNKS})
    unresolved: list[str] = [
        name for name, index in FISHEYE_FRAME_INDICES.items() if index is None
    ]
    if unresolved:
        raise ValueError(f"Discover or enter the fisheye frame indices for: {unresolved}")
    populate_fisheye_calibration_images(
        FISHEYE_RAW_CHUNKS,
        {name: int(index) for name, index in FISHEYE_FRAME_INDICES.items()},
        overwrite=OVERWRITE_FISHEYE_IMAGES,
    )

# `data/flatFieldingFunction/`

**What this is:** Raw Bayer frames used to estimate the spatial sensitivity imposed by the fisheye lens. The camera pointed at the nominally uniform Fels Planetarium dome and was rotated about its optical axis. Averaging frames across orientations reduces dome-specific spatial structure while preserving camera/lens structure.

**Where it comes from:** World-camera chunks stored in the lab Dropbox under `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/fielding_function/planetarium_fielding_function_raw`. These are the paths and selections recovered from `code/library/matlabIO/python_libraries/scratch2.ipynb`.

**What it does:** Converts the chunks to a temporary grayscale video with digital gain applied, verifies the documented 180 fps frame rate, and extracts the 36 time samples currently consumed by `defineFlatFieldingFunction.m`. Files are named sequentially in selection order, not by original video-frame number.

**Output:** `data/flatFieldingFunction/rawFrames/0.tiff` through `35.tiff`. `defineFlatFieldingFunction.m` linearizes and averages these frames, fits the flattened Gaussian, and writes `derived/flatFieldingFunction.mat`.

In [ ]:
RUN_FLAT_FIELD_EXTRACTION: bool = False
OVERWRITE_FLAT_FIELD_FRAMES: bool = False
FLAT_FIELD_RAW_CHUNKS: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "fielding_function/planetarium_fielding_function_raw"
)
FLAT_FIELD_OUTPUT: Path = DATA_ROOT / "flatFieldingFunction" / "rawFrames"

FLAT_FIELD_EXPECTED_FPS: float = 180.0
FLAT_FIELD_TIMES_SECONDS: list[int] = [
    366, 372, 393, 401, 427, 435, 456, 466, 488, 498, 518, 528, 550, 560,
    582, 592, 614, 624, 646, 656, 678, 686, 708, 718, 742, 752, 774, 784,
    806, 814, 836, 846, 868, 878, 912, 920,
]
FLAT_FIELD_FRAME_INDICES: list[int] = [
    65880, 66960, 70740, 72180, 76860, 78300, 82080, 83880, 87840,
    89640, 93240, 95040, 99000, 100800, 104760, 106560, 110520, 112320,
    116280, 118080, 122040, 123480, 127440, 129240, 133560, 135360,
    139320, 141120, 145080, 146520, 150480, 152280, 156240, 158040,
    164160, 165600,
]
assert [round(time * FLAT_FIELD_EXPECTED_FPS) for time in FLAT_FIELD_TIMES_SECONDS] == FLAT_FIELD_FRAME_INDICES

In [ ]:
def populate_flat_field_frames(raw_chunks: Path, *, overwrite: bool = False) -> None:
    """Extract the documented 36 planetarium orientations from raw chunks."""
    # Reject a stale source path before importing video or MATLAB dependencies.
    require_existing_directories({"flat-field raw chunks": raw_chunks})
    raw_chunks = raw_chunks.resolve()
    video_io: Any = load_video_io()

    # The AVI is an intermediate only. Keeping it in a temporary directory avoids
    # creating a second large calibration artifact in either Dropbox or data/.
    with tempfile.TemporaryDirectory(prefix="populate_flat_field_") as temporary_dir:
        temporary_video: Path = Path(temporary_dir) / "planetarium_fielding_function.avi"
        # Reconstruct the contiguous recording and apply its per-frame digital gain,
        # matching the preprocessing used for the checked-in flat-field TIFFs.
        video_io.world_chunks_to_video(
            str(raw_chunks),
            output_path=str(temporary_video),
            verbose=True,
            convert_to_seconds=True,
            fill_missing_frames=True,
            apply_digital_gain=True,
        )
        # The documented times map to fixed indices only at the original 180 fps.
        frames_per_second: float = float(
            video_io.inspect_video_FPS(str(temporary_video))
        )
        if not np.isclose(frames_per_second, FLAT_FIELD_EXPECTED_FPS):
            raise ValueError(
                f"Expected a {FLAT_FIELD_EXPECTED_FPS:g} fps flat-field video; "
                f"found {frames_per_second:g} fps. Review the documented frame selection."
            )
        # Convert the human-readable time selection to video indices and cross-check
        # it against the authoritative list in defineFlatFieldingFunction.m.
        frame_indices: list[int] = [
            round(time * frames_per_second) for time in FLAT_FIELD_TIMES_SECONDS
        ]
        if frame_indices != FLAT_FIELD_FRAME_INDICES:
            raise ValueError("The derived flat-field indices no longer match defineFlatFieldingFunction.m.")

        # Extract in selection order and store the results directly as 0.tiff-35.tiff.
        frames: np.ndarray = video_io.extract_frames_from_video(
            str(temporary_video), frame_indices, verbose=True, is_grayscale=True
        )
        write_tiff_stack(frames, FLAT_FIELD_OUTPUT, overwrite=overwrite)


if RUN_FLAT_FIELD_EXTRACTION:
    populate_flat_field_frames(
        FLAT_FIELD_RAW_CHUNKS, overwrite=OVERWRITE_FLAT_FIELD_FRAMES
    )

# `data/radiometricCorrectionRGB/`

**What this is:** A paired calibration between the spectral radiance of a cloudy sky measured with a PR670 and raw IMX219 images of that same sky. It is used to derive multiplicative RGB radiometric weights.

**Where it comes from:** The world-camera chunks are in Dropbox under `FLIC_data/LightLoggerRadCal/W1P1M1/radiometricCorrectionRGB/cloudyDayRecording`. The notebook logic comes from `code/preprocessRecordingData/another_scratch.ipynb`. The PR670 file `CloudySkySPD_37degSolarElevation.mat` and the illustrative `cropExample.tiff` are separately archived measurement inputs; this notebook does not recreate them.

**What it does:** Builds a temporary video without digital-gain, response-linearization, or color-weight corrections, then extracts grayscale frames 8000 through 8009. Preserving the uncorrected sensor values is essential because the downstream calibration is estimating those corrections.

**Output:** `data/radiometricCorrectionRGB/rawFrames/0.tiff` through `9.tiff`. `defineRadiometricWeights.m` combines these frames with the PR670 SPD and writes `derived/radiometricCorrectionRGB.mat`.

In [ ]:
RUN_RADIOMETRIC_FRAME_EXTRACTION: bool = False
OVERWRITE_RADIOMETRIC_FRAMES: bool = False
RADIOMETRIC_RAW_CHUNKS: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_data/LightLoggerRadCal/W1P1M1/"
    "radiometricCorrectionRGB/cloudyDayRecording"
)
RADIOMETRIC_OUTPUT: Path = DATA_ROOT / "radiometricCorrectionRGB" / "rawFrames"
RADIOMETRIC_FRAME_INDICES: list[int] = list(range(8000, 8010))


def populate_radiometric_frames(raw_chunks: Path, *, overwrite: bool = False) -> None:
    """Extract ten uncorrected cloudy-sky frames for RGB radiometric fitting."""
    # Verify the configured archive before importing video or MATLAB dependencies.
    require_existing_directories({"radiometric raw chunks": raw_chunks})
    raw_chunks = raw_chunks.resolve()
    video_io: Any = load_video_io()

    # Use a disposable AVI so the reconstructed recording never becomes a data asset.
    with tempfile.TemporaryDirectory(prefix="populate_radiometric_") as temporary_dir:
        temporary_video: Path = Path(temporary_dir) / "cloudy_day_recording.avi"
        # Disable every camera correction because these frames are inputs used to
        # estimate those corrections downstream.
        video_io.world_chunks_to_video(
            str(raw_chunks),
            str(temporary_video),
            apply_digital_gain=False,
            convert_to_seconds=True,
            fill_missing_frames=True,
            verbose=True,
            linearize_camera_responsivity=False,
            apply_color_weights=False,
        )
        # Preserve the Bayer mosaic as grayscale and rename frames by selection order.
        frames: np.ndarray = video_io.extract_frames_from_video(
            str(temporary_video), RADIOMETRIC_FRAME_INDICES, is_grayscale=True
        )
        write_tiff_stack(frames, RADIOMETRIC_OUTPUT, overwrite=overwrite)


if RUN_RADIOMETRIC_FRAME_EXTRACTION:
    populate_radiometric_frames(
        RADIOMETRIC_RAW_CHUNKS, overwrite=OVERWRITE_RADIOMETRIC_FRAMES
    )

# `data/darkSignal/`

**What this is:** Raw Bayer dark frames acquired with the lens cap installed, the camera wrapped in a black shroud, and the room dark. Separate recordings cover five fixed AGC states because exposure and analog gain can change the camera's dark behavior.

**Where it comes from:** The canonical recordings are stored in Dropbox at `FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/darkNoiseCalibrations`, which is the configured source below. Each child directory must be one raw world-camera chunk recording and should retain its metadata-rich `AGCstate*_AGain-*_DGain-*_E-*` name.

**What it does:** Counts the raw frames physically present in each state recording, chooses ten global raw indices evenly across that count, and extracts those Bayer arrays directly from their chunk files. The selected indices are written into this measurement's README. No video conversion, gain, response linearization, or color weighting is applied.

**Output:** One ten-frame directory per state beneath `data/darkSignal/`. The README is preserved. `defineDarkSignal.m` subsequently computes the median dark signal and writes `derived/darkSignal.mat`.

In [ ]:
RUN_DARK_SIGNAL_EXTRACTION: bool = False
OVERWRITE_DARK_SIGNAL_FRAMES: bool = False
DARK_SIGNAL_RAW_ROOT: Path = Path(
    "/Users/zacharykelly/Library/CloudStorage/Dropbox-Aguirre-BrainardLab/"
    "Zachary Kelly/FLIC_admin/Equipment/ArduCam B0392 IMX219 Wide Angle M12/"
    "darkNoiseCalibrations"
)
DARK_SIGNAL_OUTPUT: Path = DATA_ROOT / "darkSignal"
DARK_SIGNAL_FRAME_COUNT: int = 10
DARK_SIGNAL_STATE_PATTERN: re.Pattern[str] = re.compile(r"^AGCstate[1-5](?:_|$)")


def populate_dark_signal_frames(raw_root: Path, *, overwrite: bool = False) -> None:
    """Extract ten uncorrected dark frames from each of five fixed AGC states."""
    # Treat raw_root as the parent of the five state recordings and fail early if absent.
    require_existing_directories({"dark-signal raw root": raw_root})
    raw_root = raw_root.resolve()

    # Select only directories whose names encode one of the expected AGC states.
    state_recordings: list[Path] = sorted(
        path
        for path in raw_root.iterdir()
        if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)
    )
    if len(state_recordings) != 5:
        raise ValueError(
            f"Expected five AGC-state recording directories under {raw_root}; found {len(state_recordings)}."
        )

    selected_indices_by_state: dict[str, list[int]] = {}
    # Process each state independently and read only the ten selected raw arrays.
    for recording_path in state_recordings:
        # Spanning 0 through count-1 samples the full acquisition uniformly.
        recording_frame_count: int = raw_world_frame_count(recording_path)
        frame_indices: list[int] = [
            int(index)
            for index in np.linspace(
                0, recording_frame_count - 1, DARK_SIGNAL_FRAME_COUNT, dtype=np.int64
            )
        ]
        frames: np.ndarray = extract_raw_world_frames(recording_path, frame_indices)
        selected_indices_by_state[recording_path.name] = frame_indices
        # Keep each AGC state in its own metadata-rich output directory.
        write_tiff_stack(
            frames, DARK_SIGNAL_OUTPUT / recording_path.name, overwrite=overwrite
        )

    # Persist the exact selection produced from each recording's raw frame count.
    rows: list[str] = [
        "| AGC-state folder | Zero-based global raw-frame indices |",
        "| --- | --- |",
    ]
    for state_name, frame_indices in selected_indices_by_state.items():
        rows.append(f"| `{state_name}` | {', '.join(map(str, frame_indices))} |")
    update_generated_readme_section(
        DARK_SIGNAL_OUTPUT / "README.md", "dark-signal-frame-indices", "\n".join(rows)
    )


if RUN_DARK_SIGNAL_EXTRACTION:
    populate_dark_signal_frames(
        DARK_SIGNAL_RAW_ROOT, overwrite=OVERWRITE_DARK_SIGNAL_FRAMES
    )

# Root-level files in `data/`

The top level of `data/` contains several MAT files rather than another directory. Only the AGC-to-illuminance file came from one of the consolidated Python notebooks.

## `empircalAGCAndIlluminance.mat`

**Source:** Raw `GKA` recordings from the 2026 scripted indoor/outdoor dataset mounted at `/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026`.

**Operation:** Run `defineMSIlluminanceToAGCLag.py` to write the shared lag under `derived/`, then process the raw recordings in memory with that fixed lag, discard each recording's initial transient samples, retain finite positive samples below the configured saturation limit, and write the matched linear-scale camera-score/illuminance point cloud.

**Output:** `deriveEmpircalAGCAndIlluminance.py` reads `derived/MSIlluminanceToAGCLag.mat` and writes `data/empircalAGCAndIlluminance.mat` directly. The file contains the MATLAB struct `empiralAGC` with the fields `cameraScoreLinear`, `msIlluminance`, and `sharedLagSeconds`. The later MATLAB piecewise log-log fit is model fitting, not data population, so it is not run here.

In [ ]:
RUN_AGC_TO_ILLUMINANCE: bool = False
OVERWRITE_AGC_TO_ILLUMINANCE: bool = False
AGC_RAW_ROOT: Path = Path("/Volumes/FLIC_raw/NEWscriptedIndoorOutdoorVideos2026")
AGC_MAXIMUM_SUBJECTS: int = 4
AGC_SUBJECTS_TO_SKIP: set[str] = {"FLIC_18"}
AGC_MAXIMUM_SATURATION_PERCENT: float = 40.0
AGC_INITIAL_SAMPLES_TO_EXCLUDE: int = 100
AGC_DATA_OUTPUT: Path = DATA_ROOT / "empircalAGCAndIlluminance.mat"
AGC_LAG_OUTPUT: Path = PROJECT_ROOT / "derived" / "MSIlluminanceToAGCLag.mat"


def natural_sort_key(path_or_name: Path | str) -> list[int | str]:
    """Split names into text and integer pieces so FLIC_2 sorts before FLIC_10."""
    # Numeric tokens become integers while text tokens compare case-insensitively.
    return [int(piece) if piece.isdigit() else piece.lower() for piece in re.split(r"(\d+)", str(path_or_name))]


def populate_agc_to_illuminance(raw_root: Path, *, overwrite: bool = False) -> None:
    """Rebuild the empirical AGC/illuminance MAT file from raw subject recordings."""
    # Validate the mounted dataset before importing or running the heavier analysis.
    require_existing_directories({"AGC raw root": raw_root})
    raw_root = raw_root.resolve()

    # Collect every activity/GKA recording for the first configured valid subjects.
    recording_paths: list[str] = []
    valid_subject_count: int = 0
    for subject_dir in sorted(raw_root.iterdir(), key=natural_sort_key):
        if valid_subject_count >= AGC_MAXIMUM_SUBJECTS:
            break
        # Ignore hidden/non-subject entries and any explicitly excluded subjects.
        if (
            not subject_dir.is_dir()
            or subject_dir.name.startswith(".")
            or subject_dir.name in AGC_SUBJECTS_TO_SKIP
        ):
            continue
        # Each activity contributes its GKA directory as one analysis recording.
        for activity_dir in sorted(subject_dir.iterdir(), key=natural_sort_key):
            if not activity_dir.is_dir() or activity_dir.name.startswith("."):
                continue
            recording_path: Path = activity_dir / "GKA"
            if not recording_path.is_dir():
                raise FileNotFoundError(recording_path)
            recording_paths.append(str(recording_path))
        valid_subject_count += 1

    if not recording_paths:
        raise ValueError(f"No AGC recording directories were found under {raw_root}.")

    # Import the derivation modules only after every selected recording path has
    # passed preflight, so a bad mount cannot start any analysis dependencies.
    derive_module_dir: Path = PROJECT_ROOT / "code" / "defineWorldCameraCalibration"
    data_prep_module_dir: Path = derive_module_dir / "dataPrep"
    require_existing_directories({
        "AGC derivation module directory": derive_module_dir,
        "AGC data-prep module directory": data_prep_module_dir,
    })
    for import_path in (derive_module_dir, data_prep_module_dir):
        if str(import_path) not in sys.path:
            sys.path.insert(0, str(import_path))
    import defineMSIlluminanceToAGCLag
    import deriveEmpircalAGCAndIlluminance
    importlib.reload(defineMSIlluminanceToAGCLag)
    importlib.reload(deriveEmpircalAGCAndIlluminance)

    # Do not replace the curated MAT output without an explicit overwrite request.
    if AGC_DATA_OUTPUT.exists() and not overwrite:
        raise FileExistsError(AGC_DATA_OUTPUT)

    # Derive the shared lag first, then process the recordings with that lag
    # and export the selected camera-score/illuminance point cloud.
    lag_result: Any = defineMSIlluminanceToAGCLag.derive_agc_lag(
        recording_paths, output_path=AGC_LAG_OUTPUT
    )
    deriveEmpircalAGCAndIlluminance.derive_empircal_agc_and_illuminance(
        recording_paths,
        lag_path=lag_result.output_path,
        output_path=AGC_DATA_OUTPUT,
        maximum_saturation_percent=AGC_MAXIMUM_SATURATION_PERCENT,
        initial_samples_to_exclude=AGC_INITIAL_SAMPLES_TO_EXCLUDE,
    )
    print(f"Generated AGC-to-illuminance data at {AGC_DATA_OUTPUT}")


if RUN_AGC_TO_ILLUMINANCE:
    populate_agc_to_illuminance(
        AGC_RAW_ROOT, overwrite=OVERWRITE_AGC_TO_ILLUMINANCE
    )

## Other root-level reference and calibration MAT files

- `ASM7341_spectralSensitivity.mat` is a reference table transcribed from the manufacturer-supplied `AS7341_Filter_Templates.xlsx` spreadsheet.
- `IMX219_spectralSensitivity.mat` is a reference table corresponding to Figure 18 of Pagnutti et al. (2017), supplied by the paper's first author.
- `camera_linearity_ND0_ND0p4_rgb_means.mat` contains the ND 0 and ND 0.4 RGB means used to fit the full-well-capacity effect. The recordings are collected with `collect_light_logger_calibration_data.m`, parsed by `convert_light_logger_calibration_data.m` with both `use_mean_frame` and `differentiate_color` enabled so the Bayer channels remain separate, and then saved by `analyze_camera_linearity_data.m`. The analysis writes into MATLAB's current directory, so the result should be reviewed before being placed in `data/`.

These files should remain curated calibration inputs rather than being silently overwritten by this notebook.

# Validate the populated data tree

This read-only audit checks the expected counts and key files after any population sections have run. It deliberately ignores deprecated artifacts, hidden Finder metadata, and Python cache files.

In [ ]:
def numbered_tiff_count(directory: Path) -> int:
    """Count sequential data TIFFs while ignoring README and hidden support files."""
    # Only numeric stems belong to the zero-based frame stacks generated above.
    return len([path for path in directory.glob("*.tiff") if path.stem.isdigit()])


checks: dict[str, bool] = {
    "six curated example images": {
        path.name for path in (DATA_ROOT / "exampleWorldCameraImages").glob("*.tiff")
    } == EXPECTED_EXAMPLE_NAMES,
    "example-image README": (DATA_ROOT / "exampleWorldCameraImages" / "README.md").is_file(),
    "example-image DGain notebook": (DATA_ROOT / "exampleWorldCameraImages" / "find_DGain.ipynb").is_file(),
    "fisheye images": numbered_tiff_count(DATA_ROOT / "fisheyeLensCalibration" / "intrinsics_calibration_images") == 47,
    "36 flat-field frames": numbered_tiff_count(FLAT_FIELD_OUTPUT) == 36,
    "radiometric frames": numbered_tiff_count(RADIOMETRIC_OUTPUT) == 10,
    "radiometric README": (DATA_ROOT / "radiometricCorrectionRGB" / "README.md").is_file(),
    "five dark-signal states": len([path for path in DARK_SIGNAL_OUTPUT.iterdir() if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)]) == 5,
    "ten frames in every dark-signal state": all(
        numbered_tiff_count(path) == 10
        for path in DARK_SIGNAL_OUTPUT.iterdir()
        if path.is_dir() and DARK_SIGNAL_STATE_PATTERN.match(path.name)
    ),
    "AGC-to-illuminance MAT": AGC_DATA_OUTPUT.is_file(),
    "AS7341 sensitivity MAT": (DATA_ROOT / "ASM7341_spectralSensitivity.mat").is_file(),
    "IMX219 sensitivity MAT": (DATA_ROOT / "IMX219_spectralSensitivity.mat").is_file(),
    "camera-linearity MAT": (DATA_ROOT / "camera_linearity_ND0_ND0p4_rgb_means.mat").is_file(),
}

for description, passed in checks.items():
    print(f"{'PASS' if passed else 'FAIL'}  {description}")

if not all(checks.values()):
    raise AssertionError("One or more expected calibration-data checks failed.")